In [18]:
import numpy as np 
import cv2 

image_path = "highway_back.png"
image = cv2.imread(image_path) 



In [2]:
# fgbg2 = cv2.createBackgroundSubtractorMOG2()
# fgbg3 = cv2.createBackgroundSubtractorKNN()

# mask1 = fgbg2.apply(image); 
# mask2 = fgbg3.apply(image)

# cv2.imshow('MOG2', mask1)
# cv2.imshow('KNN', mask2)

In [19]:
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)
for _ in range (10): 
    bg_subtractor.apply(image)
    



In [4]:
# test_image_path = "test_back.png"
# test_image = cv2.imread(test_image_path)
# fg_mask = bg_subtractor.apply(test_image)

In [5]:
# cv2.imshow("frame", fg_mask)

In [6]:
# stream_url = "https://s24.ipcamlive.com/streams/18gx7p3otabpjzrye/stream.m3u8"
stream_video = "highway_attika.mp4"

In [7]:
cap = cv2.VideoCapture(stream_video)

if not cap.isOpened():
    print("Failed to open stream.")
    exit(0)

In [ ]:
# Get frame width and height
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('mask_background.mp4', fourcc, 20.0, (frame_width, frame_height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    mask = bg_subtractor.apply(frame) 
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    contours, hierarchy = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    cv2.drawContours(mask, contours, -1, (0, 255, 0), 3)
    colored_mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    out.write(colored_mask)  # Save frame
    cv2.imshow("Stream", colored_mask)

    # motion_pixels = cv2.countNonZero(mask)
    # if motion_pixels > 500:  # You choose the threshold based on scene
    #     run_yolo = True
    # else:
    #     run_yolo = False
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

# Set Regions of Interest for the Roads. 

In [11]:
def getLineCordinates(frame, parameters):
        slope, intercept = parameters
        # Sets initial y-coordinate as height from top down (bottom of the frame)
        y1 = frame.shape[0]
        # Sets final y-coordinate as 150 above the bottom of the frame
        y2 = int(y1 - 150)
        # Sets initial x-coordinate as (y1 - b) / m since y1 = mx1 + b
        x1 = int((y1 - intercept) / slope)
        # Sets final x-coordinate as (y2 - b) / m since y2 = mx2 + b
        x2 = int((y2 - intercept) / slope)
        return np.array([x1, y1, x2, y2])

In [10]:
def findTracks(frame, lines):
    # Empty arrays to store the coordinates of the left and right lines
    left = []
    right = []
    # Loops through every detected line
    for line in lines:
        # Reshapes line from 2D array to 1D array
        x1, y1, x2, y2 = line.reshape(4)
        # Fits a linear polynomial to the x and y coordinates and returns a vector of coefficients which describe the slope and y-intercept
        parameters = np.polyfit((x1, x2), (y1, y2), 1)
        slope = parameters[0]
        y_intercept = parameters[1]
        # If slope is negative, the line is to the left of the lane, and otherwise, the line is to the right of the lane
        if slope < 0:
            left.append((slope, y_intercept))
        else:
            right.append((slope, y_intercept))
    # Averages out all the values for left and right into a single slope and y-intercept value for each line
    left_avg = np.average(left, axis=0)
    right_avg = np.average(right, axis=0)
    # Calculates the x1, y1, x2, y2 coordinates for the left and right lines
    left_line = getLineCordinates(frame, left_avg)
    right_line = getLineCordinates(frame, right_avg)
    return np.array([left_line, right_line])

def displayTrackLines(frame, lines):
        # Creates an image filled with zero intensities with the same dimensions as the frame
        lines_visualize = np.zeros_like(frame)
        # Checks if any lines are detected
        if lines is not None:
            for x1, y1, x2, y2 in lines:
                # Draws lines between two coordinates with green color and 5 thickness
                cv2.line(lines_visualize, (x1, y1), (x2, y2), (0, 255, 0), 5)
        return lines_visualize

In [ ]:
import numpy as np 
import cv2 

image_path = "highway_back.png"
image = cv2.imread(image_path) 
image = cv2.resize(image, (640, 640))
road_region = image[320:, :] 
hsv = cv2.cvtColor(road_region, cv2.COLOR_BGR2HSV)

lower_gray = np.array([0, 0, 40])
upper_gray = np.array([180, 40, 200])
road_mask = cv2.inRange(hsv, lower_gray, upper_gray)

kernel = np.ones((5,5), np.uint8)
road_mask = cv2.morphologyEx(road_mask, cv2.MORPH_OPEN, kernel)

edges = cv2.Canny(road_region, 50, 150)

road_edges = cv2.bitwise_and(edges, road_mask)

masked_image = cv2.bitwise_and(road_region, road_region, mask=road_edges)
lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=80, minLineLength=50, maxLineGap=10)
lines_visualize = findTracks(frame, lines)
lines_visualize = displayTrackLines(frame, lines)
cv2.imshow("Road Region", masked_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# create Regions : 
from shapely.geometry import Polygon
from shapely.geometry.point import Point
import numpy as np 
import cv2 

image_path = "highway_back.png"
image = cv2.imread(image_path) 

# Resize the image while preserving aspect ratio
original_height, original_width = image.shape[:2]

new_width = 640 
new_height = 640

image = cv2.resize(image, (new_width, new_height))

region = {
    "name": "Road Polygon Region",
    "polygon": Polygon([(104, 639), (330, 639), (330, 3), (104, 332)]),  # Polygon points
    "counts": 0,
    "dragging": False,
    "region_color": (255, 42, 4),  # BGR Value
    "text_color": (255, 255, 255),  # Region Text Color
}



region_label = str(region["counts"])
region_color = region["region_color"]
region_text_color = region["text_color"]
polygon_coords = np.array(region["polygon"].exterior.coords, dtype=np.int32)
polygon_coords = polygon_coords.reshape((-1, 1, 2))
centroid_x, centroid_y = int(region["polygon"].centroid.x), int(region["polygon"].centroid.y)
text_size, _ = cv2.getTextSize(
    region_label, cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.7, thickness=2
)
text_x = centroid_x - text_size[0] // 2
text_y = centroid_y + text_size[1] // 2

cv2.rectangle(image,(text_x - 5, text_y - text_size[1] - 5),(text_x + text_size[0] + 5, text_y + 5),region_color,-1,)
cv2.putText(image, region_label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, region_text_color, 2)
cv2.polylines(image, [polygon_coords], isClosed=True, color=region_color, thickness=2)

print("Image shape:", image.shape)
print("Polygon coords:", polygon_coords)  

cv2.imshow("Road Region", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

Image shape: (640, 640, 3)
Polygon coords: [[[104 639]]

 [[330 639]]

 [[330 323]]

 [[104 323]]

 [[104 639]]]
